In [2]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.functions import monotonically_increasing_id

# -------------------------
# Spark Session
# -------------------------
os.environ["HADOOP_USER_NAME"] = "root"

spark = SparkSession.builder \
    .appName('Healthcare_ETL_StarSchema') \
    .master('yarn') \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-namenode:9000") \
    .config("spark.hadoop.yarn.resourcemanager.hostname", "resourcemanager") \
    .config("spark.hadoop.yarn.resourcemanager.address", "resourcemanager:8032") \
    .config("spark.hadoop.yarn.resourcemanager.scheduler.address", "resourcemanager:8030") \
    .config("spark.driver.host", "172.30.1.13") \
    .config("spark.driver.bindAddress", "0.0.0.0") \
    .config("spark.executor.memory", "512m") \
    .config("spark.yarn.am.memory", "512m") \
    .getOrCreate()

print("Spark Started")

# -------------------------
# Paths
# -------------------------
BRONZE_PATH = "hdfs://hadoop-namenode:9000/user/jovyan/bronze/healthcare/"
GOLD_PATH = "hdfs://hadoop-namenode:9000/user/root/datalake/gold/"

# -------------------------
# Read Bronze
# -------------------------
df = spark.read.parquet(BRONZE_PATH)

# =========================================================
# CLEANING
# =========================================================
df = df.withColumn("event_time", F.to_timestamp("event_time"))

# cast numeric fields
df = df.withColumn("Age", F.col("Age").cast("int")) \
     .withColumn("Billing Amount", F.col("Billing Amount").cast("double"))

# remove nulls in critical fields
df = df.dropna(subset=[
    "patient_id",
    "Hospital",
    "Medical Condition",
    "event_time"
])

# trim text columns
df = df.withColumn("Hospital", F.trim("Hospital")) \
       .withColumn("Medical Condition", F.trim("Medical Condition"))

# =========================================================
# TRANSFORMATION (FEATURE ENGINEERING)
# =========================================================

# date key
df = df.withColumn("date_key", F.date_format("event_time", "yyyyMMdd").cast("int"))

# time features
df = df.withColumn("hour", F.hour("event_time")) \
       .withColumn("day_name", F.date_format("event_time", "EEEE"))

# age group
df = df.withColumn(
    "age_group",
    F.when(F.col("Age") < 18, "Child")
     .when(F.col("Age") < 60, "Adult")
     .otherwise("Senior")
)

# high cost flag
df = df.withColumn(
    "high_cost",
    F.when(F.col("Billing Amount") > 5000, 1).otherwise(0)
)

# admission severity
df = df.withColumn(
    "admission_severity",
    F.when(F.col("Admission Type") == "Emergency", "High")
     .when(F.col("Admission Type") == "Urgent", "Medium")
     .otherwise("Low")
)

# =========================================================
# DIM: PATIENT
# =========================================================
dim_patient = df.select(
    "patient_id",
    "Age",
    "Gender",
    "age_group"
).dropDuplicates(["patient_id"]) \
 .withColumn("patient_key", monotonically_increasing_id())

# =========================================================
# DIM: HOSPITAL
# =========================================================
dim_hospital = df.select("Hospital") \
    .dropDuplicates() \
    .withColumnRenamed("Hospital", "hospital_name") \
    .withColumn("hospital_key", monotonically_increasing_id())

# =========================================================
# DIM: CONDITION
# =========================================================
dim_condition = df.select("Medical Condition") \
    .dropDuplicates() \
    .withColumnRenamed("Medical Condition", "condition_name") \
    .withColumn("condition_key", monotonically_increasing_id())

# =========================================================
# FACT TABLE
# =========================================================
fact = df \
    .join(dim_patient, "patient_id", "left") \
    .join(dim_hospital, df.Hospital == dim_hospital.hospital_name, "left") \
    .join(dim_condition, df["Medical Condition"] == dim_condition.condition_name, "left")

fact_healthcare = fact.select(
    "patient_key",
    "hospital_key",
    "condition_key",
    "date_key",
    "hour",
    "Admission Type",
    "admission_severity",
    "Billing Amount",
    "high_cost",
    "Test Results"
)

# =========================================================
# WRITE TO GOLD
# =========================================================
print("Writing Gold Layer...")

dim_patient.write.mode("overwrite").parquet(GOLD_PATH + "dim_patient")
dim_hospital.write.mode("overwrite").parquet(GOLD_PATH + "dim_hospital")
dim_condition.write.mode("overwrite").parquet(GOLD_PATH + "dim_condition")

fact_healthcare.write.mode("overwrite").parquet(GOLD_PATH + "fact_healthcare")

print("Star Schema Completed Successfully")

spark.stop()

Spark Started
Writing Gold Layer...
Star Schema Completed Successfully
